# Batch Scoring - Energy Grid Outage Prediction

Scores every substation-day with the trained RandomForest model to produce outage risk predictions and a per-substation risk summary.

Reads from: gold_ml_features and the saved model at Files/models/outage_prediction_rf
Writes to: gold_ml_predictions, gold_ml_summary

In [ ]:
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count,
    sum as spark_sum, round as spark_round, isnan, udf
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassificationModel

# Load the model saved by the training notebook
model = RandomForestClassificationModel.load('Files/models/outage_prediction_rf')
df = spark.read.table('gold_ml_features')
print(f'Scoring {df.count()} feature rows')

In [ ]:
# Clean nulls / NaN and rebuild the exact feature set used during training
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

exclude = ('had_outage', 'event_count', 'total_affected', 'total_event_duration')
feature_cols = [
    c for c, dtype in df.dtypes
    if dtype in ('double', 'float', 'int', 'bigint', 'long') and c not in exclude
]
print(f'Features ({len(feature_cols)}): {feature_cols}')

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(df)

In [ ]:
# Score every row and derive outage probability + risk level
scored = model.transform(model_df)

extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('outage_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_outage', col('prediction').cast('int'))
    .withColumn('risk_level',
        when(col('outage_probability') > 0.8, 'critical')
        .when(col('outage_probability') > 0.6, 'high')
        .when(col('outage_probability') > 0.4, 'medium')
        .otherwise('low')
    )
    .withColumn('scored_at', current_timestamp())
    .select(
        'substation_id', 'region', 'sensor_date',
        'avg_voltage', 'avg_frequency', 'avg_power_factor', 'avg_load', 'avg_temp',
        'voltage_deviation', 'freq_deviation',
        'had_outage', 'predicted_outage',
        'outage_probability', 'risk_level',
        'scored_at'
    )
)

predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count()} rows')
predictions.groupBy('risk_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-substation risk summary
summary = (
    predictions
    .groupBy('substation_id', 'region')
    .agg(
        spark_round(avg('outage_probability'), 4).alias('avg_outage_risk'),
        spark_sum('predicted_outage').alias('predicted_outage_days'),
        count('*').alias('total_days'),
        spark_round(avg('voltage_deviation'), 2).alias('avg_voltage_deviation'),
        spark_round(avg('freq_deviation'), 4).alias('avg_freq_deviation'),
        spark_round(avg('avg_load'), 2).alias('avg_daily_load'),
    )
    .withColumn('outage_rate', spark_round(col('predicted_outage_days') / col('total_days') * 100, 1))
    .withColumn('overall_risk',
        when(col('avg_outage_risk') > 0.6, 'high')
        .when(col('avg_outage_risk') > 0.3, 'medium')
        .otherwise('low')
    )
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_outage_risk').desc())
)

summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Substation risk summary: {summary.count()} substations')
summary.show(10, truncate=False)